# Use case 1 (research): rebuild my own first paper, from prompts only

**You drive the agent. It writes the code.** This notebook holds the prompts and nothing else. Paste each one into Claude Code with this notebook open, let it fill in the empty cell below, then apply the check before moving on.

In 2015, as a second-year PhD student, my first paper was [ARGO](https://www.pnas.org/doi/10.1073/pnas.1515373112) (Yang, Santillana, Kou, *PNAS* 2015): use Google search volume, plus the disease's own history, to nowcast influenza. It took the better part of a year. We are going to rebuild the same idea on dengue in Mexico, in four prompts, without writing any Python.

Data: `../data/MX_Dengue_trends.csv` (Mexico, monthly, 2004-2011).

> Stuck, or your agent is not set up? The worked version with every output is in
> [`01_research_dengue_soln.ipynb`](01_research_dengue_soln.ipynb). You will not be blocked.

# Prompt 0: data scraping. Download all available Google search interest for "dengue", "sintomas de dengue" and "mosquito"
in Mexico over the past five years. Save it as a tidy CSV, plot all three, and tell
me the date of the last data point.
download all available monthly dengue case counts in mexico from opendengue. 

compare them with pre-fetched data in data/MX_Dengue_trends.csv.

## Prompt 1: get oriented

> use the newly downloaded monthly reported dengue cases
> in Mexico; the other columns are Google search interest for Spanish dengue terms. Load it,
> tell me the date range and the number of months, and plot cases against the `dengue` search
> series on a twin axis. Print the correlation of every search column with cases.*

Notice what the prompt does **not** say: no pandas, no `parse_dates`, no matplotlib. Describe the outcome and let the agent choose the plumbing.

**Your check, not the agent's:** do the seasonal peaks land where dengue season actually is? If they landed in February, stop and find out why.

In [ ]:
# Prompt 1: paste the agent's code here and run it.


## Prompt 2: the honest baseline

Before anything clever, reproduce what a reasonable person would have done in 2011.

> *Fit ordinary least squares of `Dengue CDC` on the single `dengue` search column, training
> only on 2004-2006. Then predict 2007-2011 from that frozen fit and report out-of-sample RMSE
> and correlation. Do not refit on anything after 2006.*

That last sentence is load-bearing. Leakage is the most common way this kind of analysis goes quietly wrong, and an agent optimizing for a good-looking number will refit if you let it.

**Your check:** the last training point is 2006. Nothing after 2006 should touch the fit.

In [ ]:
# Prompt 2: paste the agent's code here and run it.


## Prompt 3: now make it ARGO

> *Now work in log space. Regress log cases on the logs of all four search terms plus three
> autoregressive lags of log cases. Use L1 regularization with cross-validated penalty, and
> retrain on a rolling 36-month window at every time step so the model only ever sees the past.
> Also fit two references on the identical rolling scheme: an autoregression-only model, and a
> search-only model. Return all predictions on the original case scale.*

This is the paragraph that used to be a year of my life. Two structural ideas carry ARGO, and both are in it: **autoregression** (yesterday's dengue predicts today's dengue) and **dynamic training** (the search-to-disease relationship drifts, so keep refitting).

In [ ]:
# Prompt 3: paste the agent's code here and run it.


## The prompt that matters most

Now stop, before you look at any result. Ask what it decided on your behalf:

> *Walk me through what you just did, line by line. Where did you have to make a choice I did
> not specify? What would break if my data were slightly different?*

Then verify the answer rather than taking it. If it mentions anything about the search columns, go and count. A one-line check beats a confident explanation:

In [ ]:
# Check whatever the agent admits to. Start here:
#   how many of the search values are exactly zero, and what share of the record is that?


Whatever you found, hold two things at once. The agent's patch was probably *correct*. It was also probably *silent*, and a silent correct patch and a silent wrong one look identical from the outside.

> This is the single lesson I would keep if I could keep only one. The agent closes the gap
> between having an idea and seeing a number, almost completely. It does not close the gap
> between seeing a number and believing it. That gap is still the job.

## Prompt 4: score everything against each other

> *Build one comparison table over the common evaluation window: RMSE, MAE, and correlation for
> the static baseline, the autoregression-only model, the search-only model, and ARGO. Add a
> column giving each model's RMSE relative to the autoregression benchmark. Then plot the ARGO
> prediction against the truth over time.*

In [ ]:
# Prompt 4: paste the agent's code here and run it.


## Now read the table honestly

The agent will not do this part for you, and it is the part that decides whether the analysis is any good. Before you look at my version, write down your own answers:

- Which single change bought the most accuracy?
- How much did the search data actually add, over and above the disease's own history?
- Is that margin big enough to justify the claim you were hoping to make?

**Then compare with [`01_research_dengue_soln.ipynb`](01_research_dengue_soln.ipynb)**, which has the executed outputs and my reading of them. Short version: dynamic training is the big win, autoregression does most of the rest, and search adds a genuine but modest amount. I say so plainly there rather than inflating my own paper, and being able to write that sentence is the skill this whole notebook is about.

---

**Next:** [`02_education_reading_group.ipynb`](02_education_reading_group.ipynb), where the agent is not writing models at all.